# 02 · Campaign — the VHH design campaign → results CSV

**Standard slot:** *design campaign.* **For Project 17 this means:** run the de novo VHH campaign —
RFantibody (RFdiffusion-Ab diffuses CDR loops onto the framework against your epitope, then ProteinMPNN
designs the loop sequence), or BoltzGen nanobody mode — score each (VHH, antigen) complex, and write a
results CSV (D2).

**Compute reality (be honest):** the real campaign wants an **A100**. A free T4 runs only a **tiny
RFantibody demo**. De novo nanobody hit rates are **LOW**, so you generate **many** (500+ where
feasible) and send survivors to a **display screen** (notebook 05) — designs are *screening inputs*,
not finished binders. This notebook runs on the **mock** backend so the plumbing executes anywhere.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Version-verify the pinned upstreams (rule 4)

Tools change. Before a real run, confirm the pinned upstream repos still exist and **pin the exact
commit** you used (put it in `LOG.md`). This HTTP-checks the URLs; it does not install anything.

In [ ]:
import requests

# Pinned upstreams for the antibody family (pin the COMMIT you actually use — these move).
UPSTREAMS = {
    "RFantibody (RFdiffusion-Ab + ProteinMPNN)": "https://github.com/RosettaCommons/RFantibody",
    "ImmuneBuilder / NanoBodyBuilder2 (IgFold-style)": "https://github.com/oxpig/ImmuneBuilder",
    "ColabFold (AF2-Multimer)": "https://github.com/sokrypton/ColabFold",
    # BoltzGen: VERIFY the current public release at course start and pin it here (URL changes).
}
for name, url in UPSTREAMS.items():
    try:
        r = requests.head(url, allow_redirects=True, timeout=10)
        print(f"[{r.status_code}] {name}\n        {url}")
    except Exception as e:
        print(f"[ERR] {name}: {e}\n        {url}")
print("\nNOTE: BoltzGen — verify the current public release/repo manually and pin it (MANUAL.md §2).")
print("Pin the exact COMMIT/tag of each tool in LOG.md before any real campaign.")

## Campaign parameters

Re-state the fixed inputs (same as notebook 01) and the campaign scale. On a real A100 run set
`N_DESIGNS` to **500+** (low hit rate ⇒ generate a large pool ⇒ filter hard ⇒ screen survivors). The
mock run uses a small N so it is fast everywhere.

In [ ]:
from antibody_tools import DEFAULT_FRAMEWORK

TAA = "HER2"
EPITOPE = "A557,A560,A579,A580,A583"     # EXAMPLE — use your verified residues from notebook 01
FRAMEWORK = DEFAULT_FRAMEWORK

# Scale: mock=small so the notebook is fast; real A100 campaign -> N_DESIGNS=500+.
N_DESIGNS = 40           # -> 500+ on A100
TOOL = "mock"            # -> "rfantibody" (A100) or "boltzgen" (verify release) on Colab

print(f"campaign: {N_DESIGNS} VHH vs {TAA} @ {EPITOPE}  (tool={TOOL})")
print("Diversity BEFORE filtering: generate many CDR variants, filter aggressively in nb 03.")

## Run the campaign (mock) → score → CSV

`design_vhh_cdrs()` generates VHH candidates (CDR1/CDR2/CDR3 on the fixed framework);
`score_designs()` fills the AF2-Multimer-ab metrics **and** the developability heuristics. Switch
`TOOL` to `"rfantibody"` / `"boltzgen"` on an A100 to run for real (the functions raise a clear,
actionable `NotImplementedError` with the TODO until then).

In [ ]:
import pandas as pd
from antibody_tools import design_vhh_cdrs, score_designs

designs = design_vhh_cdrs(TAA, EPITOPE, framework=FRAMEWORK, n=N_DESIGNS, tool=TOOL)
score_designs(designs, tool=TOOL)

rows = [d.as_row() for d in designs]
camp = pd.DataFrame(rows)
# Keep the columns the filter + analysis need; drop the bulky notes/list fields for the CSV view.
cols = ["design_id", "tool", "antigen", "framework", "cdr1", "cdr2", "cdr3",
        "plddt", "pae_interaction", "scrmsd", "cdr_geom",
        "tap_score", "camsol_like", "humanness", "synthetic"]
camp = camp[[c for c in cols if c in camp.columns]]
camp["cdr3_len"] = camp["cdr3"].str.len()
camp.to_csv("results/campaign.csv", index=False)
print("wrote results/campaign.csv", camp.shape)
print("SYNTHETIC?" , bool(camp["synthetic"].all()), "(mock => all numbers are EXAMPLE_DATA)")
camp.head()

## Quick campaign sanity look

Before filtering, eyeball the distributions: CDR3 length, the key interface metric (pae_interaction),
and the developability heuristics. On the **mock** backend these are SYNTHETIC and only show the
plumbing; on a real run they tell you whether the pool is diverse and worth filtering.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 3, figsize=(11, 3))
ax[0].hist(camp["cdr3_len"], bins=12); ax[0].set_title("CDR3 length"); ax[0].set_xlabel("aa")
ax[1].hist(camp["pae_interaction"], bins=12); ax[1].set_title("pae_interaction (mock)"); ax[1].set_xlabel("Å")
ax[2].hist(camp["humanness"], bins=12); ax[2].set_title("humanness (heuristic)"); ax[2].set_xlabel("0–1")
plt.suptitle("Campaign pool — SYNTHETIC (mock) distributions; for plumbing only")
plt.tight_layout(); plt.savefig("results/campaign_distributions.png", dpi=150); plt.show()
print("Reminder: mock distributions are SYNTHETIC — real shape comes from the A100 campaign.")

## D2 checklist
- [ ] `results/campaign.csv`: the VHH pool (CDRs + metrics), one row per design.
- [ ] Version-verify cell run; exact tool **commits** pinned in `LOG.md`.
- [ ] Design log: framework, epitope, N, seed, tool/version, runtime per design.
- [ ] (Real run) campaign at **500+** on A100; note the realistic LOW hit rate and that survivors go
      to a display screen, not straight to "binder".
- [ ] 3–4 page interim report.

**Next:** `03_filter_and_rank.ipynb` — the shared antibody filter.